# 02 - Merging into daily, seasonal and yearly tables

**Goal:** combine the cleaned tables into the datasets the analysis and the model use.

* **Daily** - one row per region per day. This is the master table.
* **Seasonal** - one row per region per Ethiopian season (Bega Oct-Jan, Belg Feb-May, Kiremt Jun-Sep).
* **Yearly** - one row per region per year.

Rain (CHIRPS) is the backbone: every other table is attached to it with a **left join**, so no rainy day is ever lost.
Where another dataset has no value (for example SMAP before 2015), the cell stays **empty**.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Find ML/Src/processing from wherever the notebook is opened (ML/Notebooks or the project root)
here = Path.cwd().resolve()
candidates = [here, here.parent, here / "ML", here.parent / "ML", *here.parents]
PROC_SRC = next((c / "Src" / "processing" for c in candidates
                 if (c / "Src" / "processing" / "pipeline_common.py").exists()), None)
assert PROC_SRC is not None, "Could not find Src/processing - open this notebook from the ML/Notebooks folder."
sys.path.insert(0, str(PROC_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline_common import (RAW_DIR, PROC_DIR, DATA_DIR, REGIONS, REGION_NAMES,
                             read_region_files, read_static_file, load_table, save_table, longest_run)
import label_config as cfg

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
COLORS = {"Arsi": "#1f77b4", "Bale": "#d62728", "Jimma": "#2ca02c"}
FIG_DIR = DATA_DIR.parent / "Figures"
FIG_DIR.mkdir(exist_ok=True)

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

print("Raw data :", RAW_DIR)
print("Processed:", PROC_DIR)
print("Figures  :", FIG_DIR)

In [ ]:
import merge_daily as md

## 1. Load the cleaned tables

In [ ]:
rain = load_table("rainfall_daily")
weather = load_table("weather_daily")
soil = load_table("soil_moisture_daily")
static = load_table("static_context", parse_dates=())

for name, t in [("rain", rain), ("weather", weather), ("soil moisture", soil)]:
    print(f"{name:<14} rows={len(t):>6}   {t['date'].min().date()} -> {t['date'].max().date()}")
print("static rows:", len(static))

The tables end on different dates because each satellite or model publishes with a different delay. The rain table sets the date range of the merged table.

## 2. Join step by step
After every join the row count must stay the same. If it grows, there are duplicates; if it shrinks, we lost days.

In [ ]:
step = rain
print("rain only        ", step.shape)
step = step.merge(weather, on=["date", "region"], how="left")
print("+ weather        ", step.shape)
step = step.merge(soil, on=["date", "region"], how="left")
print("+ soil moisture  ", step.shape)
keep = [c for c in md.STATIC_KEEP if c in static.columns]
step = step.merge(static[["region"] + keep], on="region", how="left")
print("+ static context ", step.shape)

In [ ]:
integrated = md.build_integrated_daily(rain, weather, soil, static)   # same logic, plus duplicate check

n_days = (integrated["date"].max() - integrated["date"].min()).days + 1
print("rows:", len(integrated), "| expected:", len(REGION_NAMES) * n_days)
assert len(integrated) == len(REGION_NAMES) * n_days, "a region-day is missing or duplicated"
print("duplicate region-days:", int(integrated.duplicated(["region", "date"]).sum()))
integrated.head()

## 3. What is empty, and when?
Rain, POWER and ERA5-Land should be nearly complete. SMAP only exists from 2015, so it is empty before that - **that is expected, not an error.**

In [ ]:
cols = ["rain_mm", "t2m_c", "vpd_kpa", "sm_l1", "smap_l3_sm", "smap_l4_rootzone"]
cover = integrated.groupby("year")[cols].apply(lambda g: g.notna().mean()).round(2)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(cover.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=30, ha="right")
ax.set_yticks(range(0, len(cover), 5), cover.index[::5])
ax.set_title("Share of days with a value, per year")
ax.grid(False)
plt.colorbar(im, label="share of days with data")
savefig("merge_data_coverage")
plt.show()

## 4. Seasonal table
One row per region per season. `complete` is True only when every day of the season was observed, so partial seasons at the end of the data are not mistaken for dry seasons.

In [ ]:
seasonal = md.build_seasonal(integrated)
display(seasonal.head(6))

ok = seasonal[seasonal["complete"]]
mean_rain = ok.groupby(["season", "region"])["rain_total_mm"].mean().unstack().loc[["bega", "belg", "kiremt"]]
ax = mean_rain.plot.bar(figsize=(8, 3.6), color=[COLORS[c] for c in mean_rain.columns], rot=0)
ax.set_title("Mean seasonal rainfall (complete seasons)")
ax.set_ylabel("mm per season")
savefig("merge_seasonal_rain")
plt.show()

## 5. Yearly table

In [ ]:
yearly = md.build_yearly(integrated)
display(yearly[yearly["complete"]].head(6))

pivot = yearly[yearly["complete"]].pivot(index="year", columns="region", values="max_dry_run_days")
ax = pivot.plot(figsize=(10, 3.5), color=[COLORS[c] for c in pivot.columns])
ax.set_title("Longest dry run of the year (days with rain < dry-day threshold)")
ax.set_ylabel("days")
plt.show()

## 6. Save
Same files the pipeline script writes, so either route gives the same result.

In [ ]:
save_table(integrated, "integrated_daily")
save_table(seasonal, "integrated_seasonal")
save_table(yearly, "integrated_yearly")

**Next:** `03_eda.ipynb` explores these tables. The labels and training dataset (`create_labels.py`, `build_features.py`) must have been run first for the second half of the EDA.